# v17 — Full Listwise Softmax (K=23, 샘플링 없음) + H200

**핵심 아이디어**: v14의 `ListwiseSoftmaxLoss`는 수학적으로 Plackett-Luce top-1 모델인데,
지금까지(v14~v16)는 VRAM 한계(H100 80GB) 때문에 23개 가능한 오답 중 7개만 distribution-informed
샘플링해서 근사해왔다(`idea.md` §5-11 — 그리드서치+실제 accelerate 검증으로 H100에서는 K=7이 물리적
한계임을 확정). v17은 **이 근사를 버리고 그룹당 24개(정답 1 + 오답 23) 전부를 한 번에 넣어 완전한
Plackett-Luce top-1 손실을 계산**한다 — "빼지 말고 더해야 한다"(v2→v4, §9)는 이 프로젝트의 가장
오래된 교훈의 논리적 종착점. H200(141GB, H100의 1.76배)에서 실행하는 것을 전제로 설계함.

**v14/v16과의 차이**: loss/LoRA/LR은 전부 v14와 동일. 유일한 변경은 negative 구성(K=7 샘플링 →
K=23 전체)이라, 단일 변수 실험 원칙을 지킨다. curriculum(v16)은 이번엔 포함하지 않음(별개 실험).

**⚠️ 주의**: group_size가 8→24로 3배가 되면서 VRAM 요구량도 크게 늘어난다. 이 노트북은 실행
서버(H200)의 실제 한계를 모르므로, §8에 VRAM 자가진단 셀을 넣어뒀다 — 본 학습 전에 반드시
먼저 돌려서 `TRAIN_MINIBATCH`가 그 서버에서 OOM 안 나는지 확인할 것.

## 1. 환경 체크

In [ ]:
import sys, os, subprocess, importlib

REQUIRED = ["torch", "transformers", "peft", "accelerate", "pandas", "PIL", "tqdm", "huggingface_hub", "kagglehub"]
missing = []
for pkg in REQUIRED:
    name = "PIL" if pkg == "PIL" else pkg
    try:
        importlib.import_module(name)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"누락된 패키지 설치 중: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
else:
    print("필요한 패키지 전부 설치되어 있음")

import torch
print(f"torch={torch.__version__}  cuda_available={torch.cuda.is_available()}")

# FlashAttention2 — 멀티이미지 어텐션 특성상 sdpa/eager 대비 속도 차이가 큼.
# 컴파일이 필요해 수 분 걸릴 수 있고 환경에 따라 실패할 수 있음 — 실패해도 sdpa로 자동 폴백되니 여기서 안 죽음.
try:
    import flash_attn
    print(f"flash-attn 이미 설치됨: {flash_attn.__version__}")
except ImportError:
    print("flash-attn 설치 시도 중... (컴파일 필요, 수 분 소요될 수 있음)")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"], check=True)
        import flash_attn
        print(f"flash-attn 설치 완료: {flash_attn.__version__}")
    except Exception as e:
        print(f"flash-attn 설치 실패 — sdpa로 폴백함(속도 저하 있을 수 있음): {e}")

## 2. Config

이 서버에서만 다르게 잡아야 하는 값은 이 셀만 수정하면 된다.

In [ ]:
from pathlib import Path

# 전부 노트북 파일 기준 상대경로 — 어느 서버에 옮겨놔도 그 자리에서 그대로 동작
PROJECT_ROOT = Path(".")
DATA_DIR     = Path("./data/snuaichallenge_data")
MODEL_LOCAL  = Path("./models/Qwen3-VL-8B-Instruct")
MODEL_HF_ID  = "Qwen/Qwen3-VL-8B-Instruct"  # 로컬에 없으면 여기서 다운로드
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
LOG_DIR      = PROJECT_ROOT / "logs"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MAX_IMAGE_SIZE = 448
VAL_RATIO      = 0.05
SEED           = 42

LORA_R, LORA_ALPHA, LORA_DROPOUT = 128, 256, 0.05
LR = 5e-5
EPOCHS = 5
BATCH_SIZE = 1
GRAD_ACCUM = 8
WARMUP_RATIO = 0.05
LOGGING_STEPS = 50

# ── v17 핵심 변경: K=7 샘플링 -> K=23(전체) ──────────────────────
# 그룹 크기가 8 -> 24로 3배가 되므로 TRAIN_MINIBATCH는 H100 때보다 훨씬 작게 잡아야 할 수 있음.
# 아래 값은 시작점일 뿐 — §8 VRAM 자가진단 셀로 이 서버(H200)에 맞는 값을 반드시 재확인할 것.
TRAIN_MINIBATCH  = 8
INFER_BATCH_SIZE = 24

CKPT_NAME = "best_v17"
print("config 로드 완료 (K=23 전체, group_size=24)")

## 3. GPU 개수 자동 감지

In [ ]:
import torch
NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS == 0:
    raise RuntimeError("GPU가 감지되지 않았습니다.")
print(f"감지된 GPU 개수: {NUM_GPUS}")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {props.name}  VRAM={props.total_memory/1e9:.1f}GB")

## 4. 모델 가중치 확인/다운로드

로컬(`./models/...`)에 이미 완전하게 있으면 그대로 쓰고, 없으면 `!hf download`
셸 명령으로 정확히 `./` 밑에만 받는다 — `HF_HOME`도 `./`로 강제 리다이렉트해서 root 홈
디렉토리는 전혀 건드리지 않음 (Qwen3-VL-8B-Instruct는 공개 모델, gated 아님).

In [ ]:
import os
# Xet 백엔드(hf_xet)는 컴퓨트 노드 방화벽이 Xet CDN 도메인을 막아두면 타임아웃 없이 멈추는 경우가 있음
os.environ["HF_HUB_DISABLE_XET"] = "1"
# 캐시/락 파일까지 전부 ./ 밑에만 쓰도록 강제 (root 홈 디렉토리 절대 사용 금지)
os.environ["HF_HOME"] = str((PROJECT_ROOT / ".cache" / "huggingface").resolve())
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
os.environ["HF_TOKEN"] = HF_TOKEN

def model_is_complete():
    index_file = MODEL_LOCAL / "model.safetensors.index.json"
    if not index_file.exists():
        return False
    import json
    weight_map = json.loads(index_file.read_text())["weight_map"]
    return all((MODEL_LOCAL / f).exists() for f in set(weight_map.values()))

if model_is_complete():
    print(f"모델 이미 존재(무결성 확인됨): {MODEL_LOCAL}")
else:
    MODEL_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    # 이전에 중간에 끊긴 시도들이 안 지워진 lock을 남겨서 새 다운로드가 무한 대기하는 경우가 있음
    _stale_locks = list(MODEL_LOCAL.rglob("*.lock"))
    if _stale_locks:
        print(f"stale lock {len(_stale_locks)}개 정리: {[str(p) for p in _stale_locks]}")
        for _lf in _stale_locks:
            _lf.unlink(missing_ok=True)
    print(f"모델 다운로드 중 -> {MODEL_LOCAL} (hf download, ./ 밑에 직접 설치)")
    !hf download {MODEL_HF_ID} --local-dir {MODEL_LOCAL} --token {HF_TOKEN}
    if not model_is_complete():
        raise RuntimeError(f"다운로드 후에도 {MODEL_LOCAL}의 safetensors 샤드가 불완전합니다 — 위 로그 확인 필요")
    print(f"다운로드 완료: {MODEL_LOCAL}")

MODEL_PATH = str(MODEL_LOCAL)

## 5. 데이터 확인/다운로드 (Kaggle API)

Kaggle 토큰은 코드 안에 직접 박아두고, 실행 시점에 표준 경로 `~/.kaggle/access_token`에
자동으로 기록해서 `kagglehub`가 인증하도록 한다. 다운로드는 `!python3 -c "..."` 셸 명령으로 실행하고,
`KAGGLEHUB_CACHE`와 `output_dir`을 둘 다 `./` 밑으로 강제해서 root 홈 디렉토리는 절대 안 씀
(root 파티션이 작아서 거기 받으면 용량 부족으로 죽는 문제가 있었음).

In [ ]:
KAGGLE_TOKEN = "YOUR_KAGGLE_API_TOKEN_HERE"
_kdir = Path.home() / ".kaggle"
_kdir.mkdir(parents=True, exist_ok=True)
_ktok = _kdir / "access_token"
_ktok.write_text(KAGGLE_TOKEN)
os.chmod(_ktok, 0o600)
# kagglehub 캐시도 전부 ./ 밑으로 강제 (root 홈 디렉토리 용량 부족 방지)
os.environ["KAGGLEHUB_CACHE"] = str((PROJECT_ROOT / ".cache" / "kagglehub").resolve())

if (DATA_DIR / "train.csv").exists() and (DATA_DIR / "test.csv").exists():
    print(f"데이터 이미 존재: {DATA_DIR}")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"데이터 다운로드 중 -> {DATA_DIR} (kagglehub, ./ 밑에 직접 설치)")
    !python3 -c "import kagglehub; kagglehub.competition_download('snuaichallenge', output_dir='{DATA_DIR}')"
    if not (DATA_DIR / "train.csv").exists():
        candidates = list(DATA_DIR.rglob("train.csv"))
        if candidates:
            DATA_DIR = candidates[0].parent
        else:
            raise FileNotFoundError(f"{DATA_DIR} 아래에서 train.csv를 못 찾았습니다 — 위 다운로드 로그를 확인하세요.")
    print(f"최종 DATA_DIR = {DATA_DIR}")

## 6. Hard Negative(전체 K=23) / Dataset / Loss / Model 정의

v14와 유일하게 다른 지점: `sample_group()`이 무작위 샘플링 없이 **가능한 오답 23개 전부**를
반환한다. 매 epoch 그룹 구성이 항상 동일(=live resampling 개념 자체가 사라짐, 뺄 것도 더할 것도
없이 이미 완전하므로).

In [ ]:
import ast, random, time
from itertools import permutations
from PIL import Image
from torch.utils.data import Dataset, DataLoader

ALL_PERMS = list(permutations([1, 2, 3, 4]))

def kendall_dist(p, q):
    rank = {v: i for i, v in enumerate(q)}
    arr = [rank[v] for v in p]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv

def sample_group(gt):
    """v17 핵심 변경: 샘플링 없이 23개 오답 전부 + 정답 1개 = 24개."""
    samples = [(list(gt), 0)]
    for p in ALL_PERMS:
        if p == gt:
            continue
        samples.append((list(p), kendall_dist(p, gt)))
    return samples

PROMPT = (
    "Sentence: {sentence}\n\n"
    "These 4 frames are presented in this exact order.\n"
    "Please carefully examine the changes between consecutive frames.\n"
    "Is this the correct chronological order of events?\n"
    "Answer only with \"Yes\" or \"No\"."
)
SYSTEM = (
    "You are a temporal ordering assistant. "
    "Given video frames in a specific order and a caption, "
    "determine if the frames are in the correct chronological order."
)

def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = MAX_IMAGE_SIZE / max(w, h)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

def build_messages(images, sentence):
    content = []
    for i, img in enumerate(images, 1):
        content.append({"type": "text", "text": f"Frame {i}:"})
        content.append({"type": "image", "image": img})
    content.append({"type": "text", "text": PROMPT.format(sentence=sentence)})
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": content}]

class GroupedTemporalDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sid = row["Id"]
        gt = tuple(ast.literal_eval(row["Answer"]))
        img_dir = DATA_DIR / "train" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]
        images_list, sentences, dists = [], [], []
        for perm, dist in sample_group(gt):
            inv = [0] * 4
            for inp_idx, t_pos in enumerate(perm):
                inv[t_pos - 1] = inp_idx
            imgs = [base_imgs[inv[t]] for t in range(4)]
            images_list.append(imgs)
            sentences.append(row["Sentence"])
            dists.append(int(dist))
        return {"images": images_list, "sentences": sentences, "dists": dists, "group_size": len(images_list)}

def collate_fn(batch):
    return {
        "images": [b["images"] for b in batch],
        "sentences": [b["sentences"] for b in batch],
        "dists": [b["dists"] for b in batch],
        "group_sizes": [b["group_size"] for b in batch],
    }

class ListwiseSoftmaxLoss:
    """v14와 동일한 정의. group_size가 24로 커진 것 외엔 로직 변경 없음
    (K=23이면 이건 근사가 아니라 완전한 Plackett-Luce top-1 likelihood)."""
    def __call__(self, logits, dists, group_sizes):
        offset = 0
        losses = []
        for gs in group_sizes:
            g_logits = logits[offset: offset + gs]
            g_dists = dists[offset: offset + gs]
            offset += gs
            pos_idxs = [i for i, d in enumerate(g_dists) if d == 0]
            if not pos_idxs:
                continue
            log_probs = torch.log_softmax(g_logits, dim=0)
            losses.append(-log_probs[pos_idxs[0]])
        return torch.stack(losses).mean()

def get_model_class(model_path):
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(model_path)
    mt = getattr(cfg, "model_type", "")
    if mt == "qwen3_vl":
        from transformers import Qwen3VLForConditionalGeneration
        return Qwen3VLForConditionalGeneration
    if mt == "qwen2_5_vl":
        from transformers import Qwen2_5_VLForConditionalGeneration
        return Qwen2_5_VLForConditionalGeneration
    from transformers import Qwen2VLForConditionalGeneration
    return Qwen2VLForConditionalGeneration

def load_model_and_processor(resume_from=None):
    from transformers import AutoProcessor
    from peft import LoraConfig, PeftModel, get_peft_model, TaskType
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    ModelClass = get_model_class(MODEL_PATH)
    model = None
    for attn_impl in ("flash_attention_2", "sdpa", "eager"):
        try:
            model = ModelClass.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16, attn_implementation=attn_impl)
            break
        except Exception:
            continue
    model.gradient_checkpointing_enable()
    if resume_from:
        model = PeftModel.from_pretrained(model, resume_from, is_trainable=True)
    else:
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            bias="none",
        )
        model = get_peft_model(model, lora_config)
    return model, processor

def get_yes_no_token_ids(processor):
    tok = processor.tokenizer
    return tok.convert_tokens_to_ids(tok.tokenize("Yes"))[-1], tok.convert_tokens_to_ids(tok.tokenize("No"))[-1]

def forward_logit(model, inputs, yes_id, no_id):
    outputs = model(**inputs)
    last_logits = outputs.logits[:, -1, :].float()
    log_probs = torch.log_softmax(last_logits, dim=-1)
    score = log_probs[:, yes_id] - log_probs[:, no_id]
    return score.clamp(-100.0, 100.0)

print("핵심 로직 정의 완료 (K=23 전체 그룹, group_size=24)")

## 7. Validation 함수

In [ ]:
def val_exact_match(model, processor, val_raw_df, yes_id, no_id, chunk_size):
    model.eval()
    correct, wrong_cases = 0, []
    for _, row in val_raw_df.iterrows():
        sid = row["Id"]
        sentence = row["Sentence"]
        gt = ast.literal_eval(row["Answer"])
        img_dir = DATA_DIR / "train" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]

        def reorder(order):
            inv = [0] * 4
            for inp_idx, t_pos in enumerate(order):
                inv[t_pos - 1] = inp_idx
            return [base_imgs[inv[t]] for t in range(4)]

        texts, imgs_list = [], []
        for perm in ALL_PERMS:
            imgs = reorder(list(perm))
            msgs = build_messages(imgs, sentence)
            text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            texts.append(text)
            imgs_list.append(imgs)

        scores_all = []
        with torch.no_grad():
            for bi in range(0, len(texts), chunk_size):
                inp = processor(text=texts[bi:bi+chunk_size], images=imgs_list[bi:bi+chunk_size],
                                 return_tensors="pt", padding=True).to(model.device)
                s = forward_logit(model, inp, yes_id, no_id)
                scores_all.extend(s.cpu().tolist())
        scores = [(sc, list(p)) for sc, p in zip(scores_all, ALL_PERMS)]
        best = max(scores, key=lambda x: x[0])[1]
        if best == gt:
            correct += 1
        else:
            diffs = [i for i in range(4) if best[i] != gt[i]]
            t = "adj_swap" if (len(diffs) == 2 and abs(diffs[0] - diffs[1]) == 1) else "other"
            wrong_cases.append(t)

    acc = correct / len(val_raw_df)
    adj = sum(1 for t in wrong_cases if t == "adj_swap")
    print(f"[Val] exact_match={acc:.4f} ({correct}/{len(val_raw_df)})  adj_swap_fail={adj}  other_fail={len(wrong_cases)-adj}")
    model.train()
    return acc

## 8. ⚠️ VRAM 자가진단 (본 학습 전 필수 실행)

group_size가 24로 커져서 H100 때 그리드서치로 확정했던 `TRAIN_MINIBATCH=8`이 이 서버(H200)에도
맞는 값인지는 다시 확인해야 한다. 실제 학습과 동일한 패턴(청크 나눠 forward, 그룹 전체 logit
모아서 backward)으로 한 그룹을 실제로 처리해보고 peak VRAM을 찍는다. **OOM이 나면
`TRAIN_MINIBATCH`를 줄이고 이 셀부터 다시 실행할 것.**

In [ ]:
def vram_selfcheck():
    import pandas as pd
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model, processor = load_model_and_processor()
    model = model.to("cuda:0")
    yes_id, no_id = get_yes_no_token_ids(processor)
    criterion = ListwiseSoftmaxLoss()

    row = pd.read_csv(DATA_DIR / "train.csv").iloc[0]
    gt = tuple(ast.literal_eval(row["Answer"]))
    img_dir = DATA_DIR / "train" / row["Id"]
    files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
    base_imgs = [load_image(str(img_dir / f)) for f in files]

    texts, imgs_list, dists = [], [], []
    for perm, dist in sample_group(gt):
        inv = [0] * 4
        for ii, tp in enumerate(perm):
            inv[tp - 1] = ii
        imgs = [base_imgs[inv[t]] for t in range(4)]
        msg = build_messages(imgs, row["Sentence"])
        text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        texts.append(text); imgs_list.append(imgs); dists.append(dist)
    group_size = len(texts)
    print(f"그룹 크기: {group_size} (기대: 24)")

    try:
        parts = []
        for bi in range(0, group_size, TRAIN_MINIBATCH):
            inp = processor(text=texts[bi:bi+TRAIN_MINIBATCH], images=imgs_list[bi:bi+TRAIN_MINIBATCH],
                             return_tensors="pt", padding=True).to("cuda:0")
            parts.append(forward_logit(model, inp, yes_id, no_id))
        logits = torch.cat(parts)
        loss = criterion(logits, dists, [group_size])
        loss.backward()
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"OK — TRAIN_MINIBATCH={TRAIN_MINIBATCH}  peak VRAM={peak:.2f}GB")
    except torch.cuda.OutOfMemoryError:
        print(f"OOM — TRAIN_MINIBATCH={TRAIN_MINIBATCH}가 이 서버엔 너무 큽니다. "
              "§2 Config에서 TRAIN_MINIBATCH를 줄이고(예: 4, 2) 이 셀부터 다시 실행하세요.")
    finally:
        del model
        torch.cuda.empty_cache()

vram_selfcheck()

## 9. 학습 함수 (`accelerate.notebook_launcher`로 멀티 GPU 실행)

In [ ]:
def train_fn():
    from datetime import timedelta
    from torch.optim import AdamW
    from transformers import get_cosine_schedule_with_warmup
    from accelerate import Accelerator
    from accelerate.utils import InitProcessGroupKwargs
    import pandas as pd, csv

    pg_kwargs = InitProcessGroupKwargs(timeout=timedelta(days=2))
    accelerator = Accelerator(gradient_accumulation_steps=GRAD_ACCUM, kwargs_handlers=[pg_kwargs])
    device = accelerator.device
    is_main = accelerator.is_main_process
    torch.manual_seed(SEED)

    train_csv_full = pd.read_csv(DATA_DIR / "train.csv")
    train_csv = train_csv_full.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_val = int(len(train_csv) * VAL_RATIO)
    val_raw = train_csv[:n_val].copy()
    trn_raw = train_csv[n_val:].copy()
    if is_main:
        val_raw.to_csv(CKPT_DIR / "_val_raw.csv", index=False)
        print(f"[v17] Processes={accelerator.num_processes}  train={len(trn_raw)}  val={len(val_raw)}  "
              f"group_size=24(K=23 전체), TRAIN_MINIBATCH={TRAIN_MINIBATCH}")

    model, processor = load_model_and_processor()
    yes_id, no_id = get_yes_no_token_ids(processor)
    criterion = ListwiseSoftmaxLoss()

    train_ds = GroupedTemporalDataset(trn_raw)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=collate_fn, num_workers=4, pin_memory=True)

    n_steps = (len(trn_raw) // GRAD_ACCUM) * EPOCHS
    n_warmup = int(n_steps * WARMUP_RATIO)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_cosine_schedule_with_warmup(optimizer, n_warmup, n_steps)
    model, optimizer, train_dl, scheduler = accelerator.prepare(model, optimizer, train_dl, scheduler)

    history = []
    best_val_acc = 0.0
    global_step = 0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        if is_main:
            print(f"\n{'='*60}\nEpoch {epoch}/{EPOCHS}\n{'='*60}")
        model.train()
        epoch_loss, n_batches = 0.0, 0

        for step, batch in enumerate(train_dl):
            with accelerator.accumulate(model):
                unwrapped = accelerator.unwrap_model(model)
                texts, imgs_list, group_offsets = [], [], []
                for grp_imgs, grp_sents, grp_dists in zip(batch["images"], batch["sentences"], batch["dists"]):
                    start = len(texts)
                    for imgs, sent in zip(grp_imgs, grp_sents):
                        msg = build_messages(imgs, sent)
                        text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
                        texts.append(text)
                        imgs_list.append(imgs)
                    group_offsets.append((start, len(grp_imgs), grp_dists))

                logit_parts = []
                for bi in range(0, len(texts), TRAIN_MINIBATCH):
                    inp = processor(text=texts[bi:bi+TRAIN_MINIBATCH], images=imgs_list[bi:bi+TRAIN_MINIBATCH],
                                     return_tensors="pt", padding=True).to(device)
                    logit_parts.append(forward_logit(unwrapped, inp, yes_id, no_id))
                logits = torch.cat(logit_parts)

                ord_dists, ord_sizes = [], []
                for start, size, grp_dists in group_offsets:
                    ord_dists.extend(grp_dists)
                    ord_sizes.append(size)

                loss = criterion(logits, ord_dists, ord_sizes)
                if not torch.isfinite(loss):
                    optimizer.zero_grad()
                    continue
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
                epoch_loss += loss.item()
                n_batches += 1
                if is_main and global_step % LOGGING_STEPS == 0:
                    print(f"  step={global_step:5d}  loss={epoch_loss/n_batches:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")

        accelerator.wait_for_everyone()
        if is_main:
            elapsed = (time.time() - t0) / 60
            avg_loss = epoch_loss / max(1, n_batches)
            print(f"\n[Epoch {epoch} 완료] {elapsed:.1f}분  avg_loss={avg_loss:.4f}")
            unwrapped = accelerator.unwrap_model(model)
            val_acc = val_exact_match(unwrapped, processor, val_raw, yes_id, no_id, INFER_BATCH_SIZE)
            history.append((epoch, avg_loss, val_acc))
            pd.DataFrame(history, columns=["epoch", "avg_loss", "val_acc"]).to_csv(LOG_DIR / "history.csv", index=False)
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                unwrapped.save_pretrained(CKPT_DIR / CKPT_NAME)
                processor.save_pretrained(CKPT_DIR / CKPT_NAME)
                print(f"  * Best 저장 (val_acc={val_acc:.4f})")
            unwrapped.save_pretrained(CKPT_DIR / (CKPT_NAME + "_last"))
            processor.save_pretrained(CKPT_DIR / (CKPT_NAME + "_last"))
        accelerator.wait_for_everyone()

    if is_main:
        print(f"\n학습 완료. Best val_acc={best_val_acc:.4f}")

print("train_fn 정의 완료")

## 10. 학습 실행

In [ ]:
from accelerate import notebook_launcher

notebook_launcher(train_fn, num_processes=NUM_GPUS, mixed_precision="bf16")

## 11. 학습 곡선 시각화

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

history_df = pd.read_csv(LOG_DIR / "history.csv")
print(history_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history_df["epoch"], history_df["avg_loss"], marker="o")
axes[0].set_title("avg_loss per epoch"); axes[0].set_xlabel("epoch")
axes[1].plot(history_df["epoch"], history_df["val_acc"], marker="o", color="green")
axes[1].set_title("val exact_match per epoch"); axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.savefig(LOG_DIR / "curves.png", dpi=120)
plt.show()

## 12. 추론 (24-permutation 전수조사) + 제출 파일 생성

In [ ]:
def run_inference(ckpt_name, out_name):
    import pandas as pd
    from peft import PeftModel
    from transformers import AutoProcessor
    base_model = get_model_class(MODEL_PATH).from_pretrained(
        MODEL_PATH, torch_dtype=torch.bfloat16, device_map={"": torch.device("cuda:0")}
    )
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    model = PeftModel.from_pretrained(base_model, str(CKPT_DIR / ckpt_name)).eval()
    yes_id, no_id = get_yes_no_token_ids(processor)

    test_df = pd.read_csv(DATA_DIR / "test.csv")
    submission = []
    for _, row in test_df.iterrows():
        sid, sentence = row["Id"], row["Sentence"]
        img_dir = DATA_DIR / "test" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]

        texts, imgs_list = [], []
        for perm in ALL_PERMS:
            inv = [0] * 4
            for k, t in enumerate(perm):
                inv[t - 1] = k
            imgs = [base_imgs[inv[t]] for t in range(4)]
            msg = build_messages(imgs, sentence)
            text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            texts.append(text)
            imgs_list.append(imgs)

        scores = []
        with torch.no_grad():
            for bi in range(0, len(texts), INFER_BATCH_SIZE):
                inp = processor(text=texts[bi:bi+INFER_BATCH_SIZE], images=imgs_list[bi:bi+INFER_BATCH_SIZE],
                                 return_tensors="pt", padding=True).to(model.device)
                s = forward_logit(model, inp, yes_id, no_id)
                scores.extend(s.cpu().tolist())
        best = max(zip(scores, [list(p) for p in ALL_PERMS]), key=lambda x: x[0])[1]
        submission.append({"Id": sid, "Answer": str(best)})

    out_path = PROJECT_ROOT / f"{out_name}.csv"
    pd.DataFrame(submission).to_csv(out_path, index=False)
    print(f"저장: {out_path} ({len(submission)}행)")
    del model, base_model
    torch.cuda.empty_cache()

run_inference(CKPT_NAME, "submission_v17_best")